# 13. Lecke - Ügynök memória Cognee Tudásgráfokkal


## Beállítás

Ez a jegyzetfüzet bemutatja, hogyan építhetünk intelligens **kódolási asszisztenst** tartós memóriával a [**Cognee**](https://www.cognee.ai/) tudásgráfok és a **Microsoft Agent Framework** (MAF) segítségével.

A Cognee az strukturálatlan szöveget strukturált, lekérdezhető tudásgráffá alakítja, amelyet vektoros beágyazások támogatnak — így az ügynököd gazdag, kapcsolatorientált hosszú távú memóriával rendelkezik.

### Amit megtanulsz
1. **Tudásgráfok építése**: Fejlesztői profilok és legjobb gyakorlatok strukturált, lekérdezhető tudássá alakítása.
2. **Cognee integrálása MAF-fal**: `@tool` függvények használata, hogy egy MAF ügynök lekérdezhesse a Cognee tudásgráfját.
3. **Munkamenet-tudatos párbeszédek**: Több kérdés kontextusának megőrzése ugyanabban a munkamenetben.
4. **Hosszú távú memória**: Fontos tudás megőrzése munkamenetek között és előhívása új beszélgetések során.

### Előfeltételek
- Python 3.9+
- Redis helyileg futtatva (`docker run -d -p 6379:6379 redis`) munkamenet-kezeléshez
- Egy LLM API kulcs (pl. OpenAI) — állítsd be a `LLM_API_KEY` értékét a `.env` fájlban
- `CACHING=true` a `.env` fájlban (szükséges a Cognee munkamenetekhez)
- Egy Azure AI Foundry projekt telepített chat modellel
- `AZURE_AI_PROJECT_ENDPOINT` és `AZURE_AI_MODEL_DEPLOYMENT_NAME` értékek a `.env` fájlban
- Azure CLI hitelesítve (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Az ügynök memória típusai

Ez a jegyzetfüzet azonos három memóriatípust vizsgálja, mint a fő 13. leckében, de a hosszú távú memória háttérként a Cogneet használja:

| Memória típus | Mechanizmus | Élettartam |
|---|---|---|
| **Munkamemória** | `agent.create_session()` (MAF) | Egyetlen beszélgetés |
| **Rövid távú** | Cognee munkamenet gyorsítótár (Redis) | Egy munkamenet |
| **Hosszú távú** | Cognee tudásgráf + vektorok | Állandó |

### Cognee memóriarendszere
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee tároló előkészítése


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## 1. rész — A Tudásbázis felépítése

Háromféle adatot használunk fel egy átfogó tudásbázis létrehozásához kódoló asszisztensünk számára:

1. **Fejlesztői profil** — személyes szakértelem és műszaki háttér
2. **Python legjobb gyakorlatai** — a Python Zenje gyakorlati útmutatókkal
3. **Történeti beszélgetések** — korábbi kérdés-válasz szekciók fejlesztők és AI asszisztensek között


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Tudásgráf megjelenítése

A Cognee képes egy interaktív HTML megjelenítést készíteni az általa kinyert entitásokból és kapcsolatokból.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memória gazdagítása a Memify segítségével

A `memify()` elemzi a tudásgráfot, és intelligens szabályokat generál — felismerve mintákat, bevált gyakorlatokat és a fogalmak közötti kapcsolatokat.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## 2. rész — MAF ügynök a Cognee eszközökkel

Most létrehozunk egy MAF ügynököt, amely a Cognee tudásgráfját képes lekérdezni `@tool` funkciókon keresztül. Ez lehetővé teszi az ügynök számára, hogy kihasználja a gráf-specifikus szemantikus keresés teljes erejét, miközben a párbeszédes kontextust munkamenetek révén fenntartja.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Munkamemória munkamenetekkel

Az `AgentSession` (amely az `agent.create_session()` révén jön létre) munkamemóriát biztosít egy munkameneten belül. Az ágens hivatkozhat korábbi üzenetekre, miközben lekérdezheti Cognee hosszú távú tudásgráfját is.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Új munkamenet — A hosszú távú memória megmarad

Új munkamenet indítása törli a munkamemóriát, de a Cognee tudásgráf még mindig elérhető. Az ügynök ugyanazt a hosszú távú tudást képes lekérni egy teljesen új beszélgetésben.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Összefoglaló

Ebben a jegyzetfüzetben egy olyan kódoló asszisztenst építettél, amely egyesíti **a MAF munkamemóriáját** (`agent.create_session()`) a **Cognee hosszú távú tudásgráfjával**.

### Amit megtanultál
1. **Tudásgráf építése**: A Cognee beolvassa a strukturálatlan szöveget, és létrehoz egy gráfot + vektoros memóriát.
2. **Gráfbővítés memify-val**: Származtatott tények és gazdagabb kapcsolatok az meglévő gráfodra építve.
3. **MAF + Cognee integráció**: A `@tool` funkciók lehetővé teszik, hogy egy MAF ügynök természetes módon lekérdezze Cognee gráfját.
4. **Munkamemória + hosszú távú memória**: Az `AgentSession` (az `agent.create_session()` segítségével) biztosítja a munkamenet kontextusát, míg a Cognee a tartós tudást.
5. **Szűrt keresés NodeSetekkel**: Célzottan egy bizonyos részhalmazt kérdezhetsz le a tudásgráfból (például csak elveket).

### Főbb tanulságok
- **A Cognee** a nyers szöveget strukturált, kapcsolatorientált memóriává alakítja — hatékonyabb, mint egy sík vektortár.
- A **`@tool` funkciók** tiszta hidat képeznek a MAF ügynökök és külső tudásrendszerek között.
- Az **`AgentSession`** (az `agent.create_session()` révén) elkülöníti az egyes beszélgetések kontextusát a hosszú életű tudástól.
- Ugyanaz a tudásgráf szolgál több munkamenetet és ügynököt is.

### Valós alkalmazások
- **Fejlesztői társsegédek**: kódáttekintés, incidenselemzés, architektúra-asszisztensek
- **Ügyfélkapcsolati társsegédek**: termékdokumentáció, GYIK, CRM jegyzetek támogatására szolgáló ügyfélszolgálati asszisztensek
- **Belső szakértő társsegédek**: irányelv, jogi vagy biztonsági asszisztensek, amelyek útmutatókat elemeznek
- **Egységes adattárolók**: strukturált és strukturálatlan adatok egy lekérdezhető gráffá egyesítése

### Következő lépések
- Kísérletezz időbeli tudatossággal a Cognee-ben
- Határozz meg egy OWL ontológiát az adott területre szabott gráfminőséghez
- Adj hozzá felhasználói visszacsatolási hurkokat a lekérdezések időbeli javításához
- Skálázd több ügynökös rendszerekre, amelyek ugyanazt a Cognee memória réteget használják


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Felelősségkizárás**:
Ez a dokumentum az AI fordítószolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítás hibákat vagy pontatlanságokat tartalmazhat. Az eredeti dokumentum anyanyelvén tekintendő az irányadó forrásnak. Kritikus információk esetén szakmai emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
